In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# C25 — Évaluation hors domaine des modèles d'émotion sur CaFE (H2, P7, E1)\n",
    "\n",
    "**Évaluation seulement** : aucune adaptation à CaFE. Les 5 modèles entraînés sur CREMA-D + RAVDESS\n",
    "(CNN, CRNN, têtes sur wav2vec 2.0, mHuBERT-147 et Whisper base) sont appliqués tels quels aux\n",
    "792 extraits de CaFE (français du Québec, 12 acteurs, surprise exclue).\n",
    "\n",
    "**H2** : « mHuBERT dépasse wav2vec 2.0 anglais sur le français ». Test principal : écart de F1\n",
    "macro mHuBERT − wav2vec 2.0 par bootstrap **apparié sur les acteurs** (12 acteurs, 1 000 tirages,\n",
    "graine 42). Complément : McNemar exact sur les extraits (il suppose des extraits indépendants,\n",
    "ce qui est faux ici : un acteur joue 78 extraits ; il ne sert que de repère). Verdict : confirmée si\n",
    "l'intervalle de l'écart est entièrement positif, infirmée s'il est entièrement négatif, non\n",
    "tranchée sinon.\n",
    "\n",
    "**Avant CaFE** : chaque modèle doit retrouver son F1 macro de test en domaine (graine 42) à ±0,01\n",
    "près. Sinon le notebook s'arrête : on ne mesure pas la chute avec un modèle qui n'est pas celui\n",
    "des chiffres publiés.\n",
    "\n",
    "Où sont les poids (C9, C11, C12) :\n",
    "- **CRNN** et **tête Whisper** : Hugging Face, à la révision figée de `configs/registry.yaml` ;\n",
    "- **têtes wav2vec 2.0 et mHuBERT** : sortie Kaggle de C11 (`head_seed42.pt` et `standardization.npz`)\n",
    "  si elle est attachée au notebook ; sinon réentraînement avec la même configuration et la\n",
    "  même graine (section f) ;\n",
    "- **CNN** : jamais sauvegardé (C12 n'a écrit que le meilleur réseau, le CRNN) → réentraînement\n",
    "  avec la même configuration et la même graine (section g).\n",
    "\n",
    "Internet activé, GPU T4. Standardisation et température : celles apprises sur l'anglais (R4) ;\n",
    "rien n'est réestimé sur CaFE."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## a. Démarrage\n",
    "\n",
    "`BRANCH` : la branche du dépôt à installer (`exp/c25-cafe`, ou `main` une fois fusionnée)."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "import sys\n",
    "\n",
    "BRANCH = \"exp/c25-cafe\"\n",
    "REPO_URL = \"https://github.com/mentalist011101/TP_SIGNAL_DS.git\"\n",
    "CLONE_DIR = \"/kaggle/working/TP_SIGNAL_DS\"\n",
    "\n",
    "!rm -rf {CLONE_DIR}\n",
    "!git clone --depth 1 --branch {BRANCH} {REPO_URL} {CLONE_DIR}\n",
    "# Kaggle tourne sous Python 3.13, au-delà de requires-python (<3.13) du dépôt.\n",
    "%pip install -q --ignore-requires-python -e {CLONE_DIR}\n",
    "\n",
    "sys.path.insert(0, CLONE_DIR + \"/src\")\n",
    "print(sys.version)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## b. ffmpeg et GPU"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "import shutil\n",
    "import time\n",
    "\n",
    "import torch\n",
    "\n",
    "if shutil.which(\"ffmpeg\") is None:\n",
    "    !apt-get -qq update && apt-get -qq install -y ffmpeg\n",
    "NOTEBOOK_START = time.time()\n",
    "if not torch.cuda.is_available():\n",
    "    raise RuntimeError(\"Aucun GPU : activer l'accélérateur GPU T4.\")\n",
    "device = torch.device(\"cuda\")\n",
    "print(\"GPU :\", torch.cuda.get_device_name(0))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## c. Jeux de données\n",
    "\n",
    "CREMA-D et RAVDESS (datasets Kaggle attachés, comme C10 et C12) et CaFE, version 48 kHz (archive\n",
    "officielle Zenodo, comme C16). Liens dans `/kaggle/working/datasets/<id>`."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "import urllib.request\n",
    "import zipfile\n",
    "from pathlib import Path\n",
    "\n",
    "INPUT_DIR = Path(\"/kaggle/input\")\n",
    "DATASETS_DIR = Path(\"/kaggle/working/datasets\")\n",
    "DOWNLOAD_DIR = Path(\"/kaggle/working/downloads\")\n",
    "CAFE_ZIP_URL = \"https://zenodo.org/records/1478765/files/CaFE_48k.zip?download=1\"\n",
    "CREMAD_EXPECTED_WAV = 7442\n",
    "RAVDESS_EXPECTED_WAV = 1440\n",
    "\n",
    "\n",
    "def find_cremad_root():\n",
    "    for audio_folder in sorted(INPUT_DIR.glob(\"**/AudioWAV\")):\n",
    "        if len(list(audio_folder.glob(\"*.wav\"))) == CREMAD_EXPECTED_WAV:\n",
    "            return audio_folder.parent\n",
    "    raise RuntimeError(\"CREMA-D introuvable : attacher le dataset ejlok1/cremad.\")\n",
    "\n",
    "\n",
    "def find_ravdess_root():\n",
    "    for candidate in sorted({actor.parent for actor in INPUT_DIR.glob(\"**/Actor_01\")}):\n",
    "        if len(list(candidate.glob(\"Actor_*/*.wav\"))) == RAVDESS_EXPECTED_WAV:\n",
    "            return candidate\n",
    "    raise RuntimeError(\"RAVDESS introuvable : attacher uwrfkaggler/ravdess-emotional-speech-audio.\")\n",
    "\n",
    "\n",
    "def download_cafe():\n",
    "    archive_path = DOWNLOAD_DIR / \"cafe.zip\"\n",
    "    extract_dir = DOWNLOAD_DIR / \"cafe\"\n",
    "    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)\n",
    "    urllib.request.urlretrieve(CAFE_ZIP_URL, archive_path)\n",
    "    with zipfile.ZipFile(archive_path) as archive:\n",
    "        archive.extractall(extract_dir)\n",
    "    archive_path.unlink()\n",
    "    return extract_dir\n",
    "\n",
    "\n",
    "roots = {\"cremad\": find_cremad_root(), \"ravdess\": find_ravdess_root(), \"cafe\": download_cafe()}\n",
    "DATASETS_DIR.mkdir(parents=True, exist_ok=True)\n",
    "for dataset_id, root in roots.items():\n",
    "    link = DATASETS_DIR / dataset_id\n",
    "    if link.is_symlink():\n",
    "        link.unlink()\n",
    "    link.symlink_to(root)\n",
    "    print(dataset_id, \"→\", root)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## d. Configuration, extraits d'émotion et CaFE\n",
    "\n",
    "Paramètres : `configs/ood.yaml` (modèles, tolérance ±0,01, paire de H2) et\n",
    "`configs/evaluation.yaml` (1 000 tirages, 95 %, graine 42). Les extraits en domaine sont ceux de C11\n",
    "et C12 (`emotion_examples` : découpage par acteur P1, 6 classes P2). Pour CaFE, le même\n",
    "`emotion_examples` : jeu réservé au test (P7), surprise exclue par `configs/labels.yaml`."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "import json\n",
    "\n",
    "import numpy as np\n",
    "\n",
    "from echolens.config import CONFIG_DIR, REPO_ROOT\n",
    "from echolens.core.data.emotions import load_emotion_labels\n",
    "from echolens.core.data.manifest import audio_path, load_manifest\n",
    "from echolens.core.data.splits import check_no_speaker_leak, load_speaker_splits, load_split_config\n",
    "from echolens.training.evaluation import EvaluationConfig\n",
    "from echolens.training.heads import emotion_examples, load_heads_config\n",
    "from echolens.training.ood_eval import load_ood_config\n",
    "\n",
    "ood = load_ood_config()\n",
    "evaluation_config = EvaluationConfig.from_yaml(CONFIG_DIR / \"evaluation.yaml\")\n",
    "heads_config = load_heads_config()\n",
    "labels = load_emotion_labels()\n",
    "class_names = list(labels.emotions)\n",
    "class_count = len(class_names)\n",
    "split_config = load_split_config()\n",
    "speaker_splits = load_speaker_splits(REPO_ROOT / \"data\" / \"splits\" / \"emotion_split.csv\")\n",
    "check_no_speaker_leak(speaker_splits)\n",
    "manifests_dir = REPO_ROOT / \"data\" / \"manifests\"\n",
    "\n",
    "# Étape 1 : CaFE, 792 extraits gardés et 12 acteurs (contrôle de l'énoncé de C25).\n",
    "cafe_all_rows = load_manifest(manifests_dir / \"cafe.csv\")\n",
    "cafe_examples = emotion_examples(cafe_all_rows, labels, speaker_splits, split_config)\n",
    "cafe_rows = [cafe_all_rows[example.position] for example in cafe_examples]\n",
    "cafe_targets = np.array([example.target for example in cafe_examples], dtype=np.int_)\n",
    "cafe_actors = [example.speaker for example in cafe_examples]\n",
    "cafe_genders = [example.gender for example in cafe_examples]\n",
    "cafe_intensities = [ood.intensity_names[row.intensity] for row in cafe_rows]\n",
    "if len(cafe_rows) != ood.cafe_expected_clips or len(set(cafe_actors)) != ood.cafe_expected_actors:\n",
    "    raise RuntimeError(f\"CaFE : {len(cafe_rows)} extraits, {len(set(cafe_actors))} acteurs\")\n",
    "missing = [row.path for row in cafe_rows if not audio_path(row, DATASETS_DIR).exists()]\n",
    "if missing:\n",
    "    raise RuntimeError(f\"CaFE : {len(missing)} audios absents, par exemple {missing[:3]}\")\n",
    "print(\"CaFE :\", len(cafe_rows), \"extraits,\", len(set(cafe_actors)), \"acteurs\")\n",
    "print(\"Classes :\", np.bincount(cafe_targets, minlength=class_count).tolist(), class_names)\n",
    "\n",
    "# Étape 2 : extraits en domaine (CREMA-D et RAVDESS), avec leur partie (P1).\n",
    "in_domain_rows = []\n",
    "in_domain_examples = []\n",
    "for dataset_id in heads_config.emotion_datasets:\n",
    "    rows = load_manifest(manifests_dir / f\"{dataset_id}.csv\")\n",
    "    for example in emotion_examples(rows, labels, speaker_splits, split_config):\n",
    "        in_domain_rows.append(rows[example.position])\n",
    "        in_domain_examples.append(example)\n",
    "split_of_clip = np.array([example.split for example in in_domain_examples])\n",
    "in_domain_targets = np.array([example.target for example in in_domain_examples], dtype=np.int_)\n",
    "print({name: int((split_of_clip == name).sum()) for name in [\"train\", \"validation\", \"test\"]})"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## e. Chiffres publiés (C9, C11, C12) et fonctions communes\n",
    "\n",
    "Pour chaque modèle : F1 macro de test **de la graine 42** (celle des poids) et moyenne des 3 graines\n",
    "(le chiffre cité dans le rapport). La vérification ±0,01 utilise la graine 42 ; la « chute » se\n",
    "mesure contre la moyenne publiée."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "from echolens.training.evaluation import (\n",
    "    accuracy,\n",
    "    apply_temperature,\n",
    "    expected_calibration_error,\n",
    "    fit_temperature,\n",
    "    macro_f1,\n",
    "    normalized_confusion,\n",
    "    per_class_f1,\n",
    "    scores_by_group,\n",
    ")\n",
    "from echolens.training.ood_eval import check_in_domain\n",
    "\n",
    "RESULTS_DIR = REPO_ROOT / \"training\" / \"results\"\n",
    "published = {}\n",
    "for model_id in [\"cnn\", \"crnn\"]:\n",
    "    content = json.loads((RESULTS_DIR / f\"emotions_family_a_{model_id}.json\").read_text(encoding=\"utf-8\"))\n",
    "    for seed_run in content[\"seeds\"]:\n",
    "        if seed_run[\"seed\"] == ood.reference_seed:\n",
    "            reference = seed_run[\"test\"]\n",
    "    published[model_id] = {\n",
    "        \"seed42_f1\": reference[\"macro_f1\"],\n",
    "        \"mean_f1\": content[\"summary\"][\"macro_f1\"][\"mean\"],\n",
    "        \"per_class\": reference[\"f1_per_class\"],\n",
    "    }\n",
    "c11 = json.loads((RESULTS_DIR / \"c11_family_b_emotions.json\").read_text(encoding=\"utf-8\"))\n",
    "seed_position = heads_config.seeds.index(ood.reference_seed)\n",
    "for model_id in [\"wav2vec2-base\", \"mhubert-147\", \"whisper-base\"]:\n",
    "    result = c11[\"models\"][model_id]\n",
    "    published[model_id] = {\n",
    "        \"seed42_f1\": result[\"macro_f1\"][\"values\"][seed_position],\n",
    "        \"mean_f1\": result[\"macro_f1\"][\"mean\"],\n",
    "        \"per_class\": result[\"f1_per_class\"],\n",
    "        \"dropout\": result[\"chosen\"][\"setting\"][\"dropout\"],\n",
    "    }\n",
    "for model_id, values in published.items():\n",
    "    print(f\"{model_id:14s} publié : graine 42 {values['seed42_f1']:.4f}, moyenne {values['mean_f1']:.4f}\")\n",
    "\n",
    "# Ce que chaque modèle produit : probabilités brutes en domaine (validation, test) et sur CaFE.\n",
    "validation_mask = split_of_clip == \"validation\"\n",
    "test_mask = split_of_clip == \"test\"\n",
    "y_validation = in_domain_targets[validation_mask]\n",
    "y_test = in_domain_targets[test_mask]\n",
    "probabilities = {}   # model_id -> {\"validation\": ..., \"test\": ..., \"cafe\": ...}\n",
    "temperatures = {}    # model_id -> (T, origine)\n",
    "weights_source = {}  # model_id -> d'où viennent les poids"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## f. Famille B : encodeurs gelés et têtes\n",
    "\n",
    "Les représentations (moyenne de chaque couche) se calculent avec `embed_manifest` de C10, la même\n",
    "fonction que pour les chiffres publiés. On les calcule pour CaFE et pour les extraits en domaine\n",
    "nécessaires : validation et test si les têtes sont retrouvées, **tout** (train compris) s'il faut\n",
    "réentraîner."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "import torch\n",
    "\n",
    "from echolens.core.models.heads import LayerWeightedHead\n",
    "from echolens.core.models.production_config import CONFIG_FILE, WEIGHTS_FILE, load_production_config\n",
    "from echolens.core.models.registry import ModelRegistry\n",
    "from echolens.core.models.storage import load_weights\n",
    "from echolens.training.embeddings import load_embeddings_config\n",
    "from echolens.training.encoders import embed_manifest, load_encoder\n",
    "from echolens.training.head_training import predict_probabilities, run_emotions\n",
    "from echolens.training.heads import LabeledSet, Standardization, standardize\n",
    "from huggingface_hub import hf_hub_download\n",
    "\n",
    "embeddings_config = load_embeddings_config()\n",
    "encoder_configs = {encoder.id: encoder for encoder in embeddings_config.encoders}\n",
    "registry = ModelRegistry.from_yaml(CONFIG_DIR / \"registry.yaml\")\n",
    "registry_entries = {entry.id: entry for entry in registry.models}\n",
    "\n",
    "\n",
    "def find_c11_head(model_id):\n",
    "    # Sortie Kaggle de C11 : .../emotions/<modèle>/head_seed42.pt et standardization.npz.\n",
    "    candidates = sorted(INPUT_DIR.glob(f\"**/emotions/{model_id}/head_seed{ood.reference_seed}.pt\"))\n",
    "    if candidates:\n",
    "        return candidates[0].parent\n",
    "    return None\n",
    "\n",
    "\n",
    "c11_folders = {model_id: find_c11_head(model_id) for model_id in [\"wav2vec2-base\", \"mhubert-147\"]}\n",
    "must_retrain = [model_id for model_id, folder in c11_folders.items() if folder is None]\n",
    "print(\"Têtes C11 retrouvées :\", {k: str(v) for k, v in c11_folders.items()})\n",
    "print(\"À réentraîner :\", must_retrain)\n",
    "needed = np.ones(len(in_domain_rows), dtype=bool)\n",
    "if not must_retrain:\n",
    "    needed = validation_mask | test_mask"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "# Étape 1 : représentations de chaque encodeur, pour CaFE et pour les extraits en domaine utiles.\n",
    "needed_positions = np.flatnonzero(needed)\n",
    "needed_rows = [in_domain_rows[position] for position in needed_positions]\n",
    "embeddings = {}   # encodeur -> {\"cafe\": (792, couches, dim), \"in_domain\": (extraits utiles, couches, dim)}\n",
    "for encoder_id in [\"wav2vec2-base\", \"mhubert-147\", \"whisper-base\"]:\n",
    "    encoder = load_encoder(encoder_configs[encoder_id], embeddings_config.sample_rate, device)\n",
    "    cafe_result = embed_manifest(cafe_rows, DATASETS_DIR, encoder)\n",
    "    in_domain_result = embed_manifest(needed_rows, DATASETS_DIR, encoder)\n",
    "    embeddings[encoder_id] = {\"cafe\": cafe_result.embeddings, \"in_domain\": in_domain_result.embeddings}\n",
    "    print(encoder_id, cafe_result.embeddings.shape, in_domain_result.embeddings.shape)\n",
    "    del encoder\n",
    "    torch.cuda.empty_cache()\n",
    "\n",
    "# Position, parmi les extraits utiles, de chaque partie.\n",
    "useful_split = split_of_clip[needed_positions]\n",
    "useful_targets = in_domain_targets[needed_positions]\n",
    "useful_validation = np.flatnonzero(useful_split == \"validation\")\n",
    "useful_test = np.flatnonzero(useful_split == \"test\")\n",
    "useful_train = np.flatnonzero(useful_split == \"train\")"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "def head_probabilities(head, stats, features):\n",
    "    standardized = standardize(features, stats)\n",
    "    return predict_probabilities(head, standardized)\n",
    "\n",
    "\n",
    "def new_head(layer_count, dimension, dropout):\n",
    "    return LayerWeightedHead(layer_count, dimension, heads_config.hidden_size, class_count, dropout)\n",
    "\n",
    "\n",
    "# Étape 2 : tête Whisper base, depuis Hugging Face (révision figée du registre).\n",
    "entry = registry_entries[\"echolens-emotion-whisper-base\"]\n",
    "repo_id = entry.weights.removeprefix(\"hf://\")\n",
    "whisper_config = load_production_config(Path(hf_hub_download(repo_id, CONFIG_FILE, revision=entry.revision)))\n",
    "whisper_weights = Path(hf_hub_download(repo_id, WEIGHTS_FILE, revision=entry.revision))\n",
    "shape = whisper_config.head\n",
    "whisper_head = load_weights(new_head(shape.layer_count, shape.dimension, shape.dropout), whisper_weights)\n",
    "whisper_stats = Standardization(\n",
    "    mean=np.asarray(whisper_config.standardization_mean, dtype=np.float64),\n",
    "    std=np.asarray(whisper_config.standardization_std, dtype=np.float64),\n",
    ")\n",
    "heads = {\"whisper-base\": (whisper_head, whisper_stats)}\n",
    "temperatures[\"whisper-base\"] = (whisper_config.calibration.temperature, \"registre / config.json (anglais)\")\n",
    "weights_source[\"whisper-base\"] = f\"Hugging Face {repo_id} @ {entry.revision[:8]}\"\n",
    "\n",
    "# Étape 3 : têtes wav2vec 2.0 et mHuBERT : sortie C11 si attachée.\n",
    "for model_id, folder in c11_folders.items():\n",
    "    if folder is None:\n",
    "        continue\n",
    "    layer_count, dimension = embeddings[model_id][\"cafe\"].shape[1:]\n",
    "    head = new_head(layer_count, dimension, published[model_id][\"dropout\"])\n",
    "    head.load_state_dict(torch.load(folder / f\"head_seed{ood.reference_seed}.pt\", map_location=\"cpu\"))\n",
    "    head.eval()\n",
    "    saved = np.load(folder / \"standardization.npz\")\n",
    "    heads[model_id] = (head, Standardization(mean=saved[\"mean\"], std=saved[\"std\"]))\n",
    "    weights_source[model_id] = f\"sortie Kaggle de C11 ({folder})\"\n",
    "\n",
    "# Étape 4 : réentraînement (même configuration, mêmes graines) pour les têtes manquantes.\n",
    "validation_from_run = {}\n",
    "for model_id in must_retrain:\n",
    "    features = embeddings[model_id][\"in_domain\"]\n",
    "    train_set = LabeledSet(features[useful_train], useful_targets[useful_train])\n",
    "    validation_set = LabeledSet(features[useful_validation], useful_targets[useful_validation])\n",
    "    test_set = LabeledSet(features[useful_test], useful_targets[useful_test])\n",
    "    run = run_emotions(train_set, validation_set, test_set, heads_config, class_count)\n",
    "    index = heads_config.seeds.index(ood.reference_seed)\n",
    "    heads[model_id] = (run.heads[index].model, run.heads[index].stats)\n",
    "    weights_source[model_id] = \"réentraîné ici (configs/heads.yaml, graine 42)\"\n",
    "    print(model_id, \"réentraîné : réglage\", run.chosen.setting)"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "# Étape 5 : probabilités brutes en domaine (validation, test) et sur CaFE, pour les 3 têtes.\n",
    "for model_id, (head, stats) in heads.items():\n",
    "    in_domain = embeddings[model_id][\"in_domain\"]\n",
    "    probabilities[model_id] = {\n",
    "        \"validation\": head_probabilities(head, stats, in_domain[useful_validation]),\n",
    "        \"test\": head_probabilities(head, stats, in_domain[useful_test]),\n",
    "        \"cafe\": head_probabilities(head, stats, embeddings[model_id][\"cafe\"]),\n",
    "    }\n",
    "    if model_id not in temperatures:\n",
    "        # T apprise sur la validation ANGLAISE (E2), jamais sur CaFE.\n",
    "        fitted = fit_temperature(y_validation, probabilities[model_id][\"validation\"].astype(np.float64), evaluation_config)\n",
    "        temperatures[model_id] = (fitted, \"ajustée sur la validation anglaise (fit_temperature)\")\n",
    "    f1 = macro_f1(y_test, probabilities[model_id][\"test\"].argmax(axis=1))\n",
    "    print(f\"{model_id:14s} F1 test en domaine {f1:.4f} (publié graine 42 : {published[model_id]['seed42_f1']:.4f})\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## g. Famille A : CRNN (Hugging Face) et CNN (réentraîné)\n",
    "\n",
    "Log-Mel de chaque extrait par `echolens.core.features` (R5), fenêtre centrale de 3 s, standardisation\n",
    "par bande de l'entraînement (R4). Le log-Mel des extraits en domaine est celui de C12 : si sa sortie\n",
    "`emotion_log_mels.npz` est attachée, on la relit ; sinon on la recalcule. Le **CNN** n'a jamais été\n",
    "sauvegardé : on le réentraîne avec `run_emotion_architecture` (même configuration, mêmes graines) et on\n",
    "garde la graine 42 à sa meilleure époque de validation, comme C12."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "from echolens.core.features import FeatureConfig, log_mel_spectrogram\n",
    "from echolens.core.frame_windows import center_windows, window_frame_count\n",
    "from echolens.core.audio import load_audio\n",
    "from echolens.core.models.family_a import SpectrogramCnn, SpectrogramCrnn\n",
    "from echolens.training.family_a import BandStatistics, load_family_a_config\n",
    "from echolens.training.family_a_emotions import (\n",
    "    compute_emotion_log_mels,\n",
    "    load_emotion_log_mels,\n",
    "    save_emotion_log_mels,\n",
    ")\n",
    "from echolens.training.family_a_emotions_training import emotion_parts, run_emotion_architecture\n",
    "from echolens.training.family_a_models import predict as predict_spectrograms\n",
    "\n",
    "feature_config = FeatureConfig.from_yaml(CONFIG_DIR / \"features.yaml\")\n",
    "family_a_config = load_family_a_config()\n",
    "window_frames = window_frame_count(family_a_config.emotions.window_s, feature_config)\n",
    "\n",
    "# Étape 1 : log-Mel des extraits en domaine (réutilise C12 s'il est attaché).\n",
    "LOG_MELS_PATH = Path(\"/kaggle/working/emotion_log_mels.npz\")\n",
    "attached = sorted(INPUT_DIR.glob(\"**/emotion_log_mels.npz\"))\n",
    "if attached:\n",
    "    log_mel_set = load_emotion_log_mels(attached[0])\n",
    "    print(\"Log-Mel de C12 relus :\", attached[0])\n",
    "else:\n",
    "    rows_by_dataset = {}\n",
    "    for dataset_id in heads_config.emotion_datasets:\n",
    "        rows_by_dataset[dataset_id] = load_manifest(manifests_dir / f\"{dataset_id}.csv\")\n",
    "    log_mel_set = compute_emotion_log_mels(\n",
    "        rows_by_dataset, DATASETS_DIR, labels, speaker_splits, split_config, feature_config\n",
    "    )\n",
    "    save_emotion_log_mels(log_mel_set, LOG_MELS_PATH)\n",
    "parts = emotion_parts(log_mel_set, window_frames, feature_config.db_floor)\n",
    "y_test_a = log_mel_set.targets[parts.test_indices]\n",
    "y_validation_a = log_mel_set.targets[parts.validation_indices]\n",
    "# Les deux familles doivent voir exactement les mêmes extraits de test, dans le même ordre.\n",
    "if not np.array_equal(y_test_a, y_test) or not np.array_equal(y_validation_a, y_validation):\n",
    "    raise RuntimeError(\"Les extraits de test ou de validation de la famille A diffèrent de ceux de C11\")\n",
    "\n",
    "# Étape 2 : log-Mel de CaFE, fenêtre centrale de 3 s.\n",
    "cafe_log_mels = []\n",
    "cafe_lengths = []\n",
    "for row in cafe_rows:\n",
    "    signal = load_audio(audio_path(row, DATASETS_DIR))\n",
    "    log_mel = log_mel_spectrogram(signal, feature_config)\n",
    "    cafe_log_mels.append(log_mel)\n",
    "    cafe_lengths.append(log_mel.shape[1])\n",
    "longest = max(cafe_lengths)\n",
    "padded = np.full((len(cafe_log_mels), feature_config.n_mels, longest), feature_config.db_floor, dtype=np.float32)\n",
    "for index, log_mel in enumerate(cafe_log_mels):\n",
    "    padded[index, :, : log_mel.shape[1]] = log_mel\n",
    "cafe_windows = center_windows(padded, np.array(cafe_lengths), window_frames, feature_config.db_floor)\n",
    "print(\"CaFE, fenêtres :\", cafe_windows.shape)"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "# Étape 3 : CRNN depuis Hugging Face.\n",
    "entry = registry_entries[\"echolens-emotion-crnn\"]\n",
    "repo_id = entry.weights.removeprefix(\"hf://\")\n",
    "crnn_config = load_production_config(Path(hf_hub_download(repo_id, CONFIG_FILE, revision=entry.revision)))\n",
    "crnn = SpectrogramCrnn(class_count, crnn_config.network)\n",
    "crnn = load_weights(crnn, Path(hf_hub_download(repo_id, WEIGHTS_FILE, revision=entry.revision)))\n",
    "crnn.to(device)\n",
    "crnn_stats = BandStatistics(\n",
    "    mean=np.asarray(crnn_config.standardization_mean, dtype=np.float32),\n",
    "    std=np.asarray(crnn_config.standardization_std, dtype=np.float32),\n",
    ")\n",
    "crnn_in_domain = {\n",
    "    \"validation\": predict_spectrograms(crnn, parts.validation.log_mels, crnn_stats, family_a_config, device).probabilities,\n",
    "    \"test\": predict_spectrograms(crnn, parts.test.log_mels, crnn_stats, family_a_config, device).probabilities,\n",
    "}\n",
    "crnn_in_domain[\"cafe\"] = predict_spectrograms(crnn, cafe_windows, crnn_stats, family_a_config, device).probabilities\n",
    "probabilities[\"crnn\"] = crnn_in_domain\n",
    "temperatures[\"crnn\"] = (crnn_config.calibration.temperature, \"registre / config.json (anglais)\")\n",
    "weights_source[\"crnn\"] = f\"Hugging Face {repo_id} @ {entry.revision[:8]}\"\n",
    "print(\"CRNN F1 test en domaine\", round(macro_f1(y_test, crnn_in_domain[\"test\"].argmax(axis=1)), 4))"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "# Étape 4 : CNN réentraîné (GPU, même configuration et mêmes graines que C12).\n",
    "start = time.time()\n",
    "cnn_run = run_emotion_architecture(\"cnn\", parts, class_count, family_a_config, device)\n",
    "print(f\"CNN réentraîné en {(time.time() - start) / 60:.1f} min, taux retenu {cnn_run.chosen_learning_rate:g}\")\n",
    "for seed_run in cnn_run.runs:\n",
    "    if seed_run.seed == ood.reference_seed:\n",
    "        cnn_seed = seed_run\n",
    "cnn_stats = cnn_seed.trained.statistics\n",
    "cnn_network = cnn_seed.trained.network\n",
    "probabilities[\"cnn\"] = {\n",
    "    \"validation\": cnn_seed.validation.probabilities,\n",
    "    \"test\": cnn_seed.test.probabilities,\n",
    "    \"cafe\": predict_spectrograms(cnn_network, cafe_windows, cnn_stats, family_a_config, device).probabilities,\n",
    "}\n",
    "fitted = fit_temperature(y_validation, probabilities[\"cnn\"][\"validation\"].astype(np.float64), evaluation_config)\n",
    "temperatures[\"cnn\"] = (fitted, \"ajustée sur la validation anglaise (fit_temperature)\")\n",
    "weights_source[\"cnn\"] = \"réentraîné ici (configs/family_a.yaml, graine 42)\"\n",
    "print(\"CNN F1 test en domaine\", round(macro_f1(y_test, probabilities[\"cnn\"][\"test\"].argmax(axis=1)), 4))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## h. Contrôle : chaque modèle retrouve son F1 en domaine (±0,01)\n",
    "\n",
    "Si un seul modèle est hors tolérance, **le notebook s'arrête** : il ne faut pas mesurer la chute avec\n",
    "un modèle qui n'est pas celui des chiffres publiés. Prévenir Luciano avec ce tableau."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "\n",
    "check_rows = []\n",
    "failures = []\n",
    "for model_spec in ood.models:\n",
    "    model_id = model_spec.id\n",
    "    measured = macro_f1(y_test, probabilities[model_id][\"test\"].argmax(axis=1))\n",
    "    target = published[model_id][\"seed42_f1\"]\n",
    "    ok = check_in_domain(measured, target, ood.in_domain_tolerance)\n",
    "    if not ok:\n",
    "        failures.append(model_id)\n",
    "    check_rows.append({\n",
    "        \"modèle\": model_id,\n",
    "        \"F1 test retrouvé\": round(measured, 4),\n",
    "        \"publié (graine 42)\": round(target, 4),\n",
    "        \"publié (moyenne 3 graines)\": round(published[model_id][\"mean_f1\"], 4),\n",
    "        \"écart\": round(measured - target, 4),\n",
    "        \"OK ±0,01\": ok,\n",
    "        \"poids\": weights_source[model_id],\n",
    "    })\n",
    "display(pd.DataFrame(check_rows))\n",
    "if failures:\n",
    "    raise RuntimeError(f\"ARRÊT : F1 en domaine hors tolérance pour {failures}. Ne pas évaluer sur CaFE ; prévenir Luciano.\")\n",
    "print(\"Tous les modèles retrouvent leur F1 de test en domaine : on peut évaluer sur CaFE.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## i. Mesures sur CaFE\n",
    "\n",
    "Pour chaque modèle : F1 macro, accuracy, matrice de confusion, ECE (avec la température apprise sur\n",
    "l'anglais, et brute), chute de F1 contre la moyenne publiée, F1 par classe, par intensité (faible,\n",
    "forte) et par genre. Intervalles à 95 % par **bootstrap sur les acteurs** : mêmes 1 000 tirages\n",
    "(graine 42) pour tous les modèles."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "from echolens.training.ood_eval import (\n",
    "    actor_bootstrap_ci,\n",
    "    draw_actor_samples,\n",
    "    f1_drop,\n",
    "    h2_verdict,\n",
    "    paired_difference,\n",
    ")\n",
    "\n",
    "samples = draw_actor_samples(cafe_actors, evaluation_config)\n",
    "print(len(samples), \"tirages d'acteurs\")\n",
    "predictions = {}\n",
    "results = {}\n",
    "for model_spec in ood.models:\n",
    "    model_id = model_spec.id\n",
    "    raw = probabilities[model_id][\"cafe\"].astype(np.float64)\n",
    "    temperature = temperatures[model_id][0]\n",
    "    calibrated = apply_temperature(raw, temperature, evaluation_config)\n",
    "    y_pred = raw.argmax(axis=1)\n",
    "    predictions[model_id] = y_pred\n",
    "    f1 = macro_f1(cafe_targets, y_pred)\n",
    "    low, high = actor_bootstrap_ci(cafe_targets, y_pred, samples, evaluation_config)\n",
    "    results[model_id] = {\n",
    "        \"family\": model_spec.family,\n",
    "        \"weights\": weights_source[model_id],\n",
    "        \"in_domain_f1_mean\": published[model_id][\"mean_f1\"],\n",
    "        \"in_domain_f1_seed42\": published[model_id][\"seed42_f1\"],\n",
    "        \"macro_f1\": f1,\n",
    "        \"macro_f1_ci\": [low, high],\n",
    "        \"accuracy\": accuracy(cafe_targets, y_pred),\n",
    "        \"f1_drop\": f1_drop(published[model_id][\"mean_f1\"], f1),\n",
    "        \"temperature\": temperature,\n",
    "        \"temperature_origin\": temperatures[model_id][1],\n",
    "        \"ece_calibrated\": expected_calibration_error(cafe_targets, calibrated, evaluation_config).ece,\n",
    "        \"ece_raw\": expected_calibration_error(cafe_targets, raw, evaluation_config).ece,\n",
    "        \"f1_per_class\": per_class_f1(cafe_targets, y_pred, class_count).tolist(),\n",
    "        \"f1_per_class_in_domain\": published[model_id][\"per_class\"],\n",
    "        \"f1_by_intensity\": scores_by_group(cafe_targets, y_pred, cafe_intensities),\n",
    "        \"f1_by_gender\": scores_by_group(cafe_targets, y_pred, cafe_genders),\n",
    "        \"confusion\": normalized_confusion(cafe_targets, y_pred, class_count).tolist(),\n",
    "    }\n",
    "table = []\n",
    "for model_id, result in results.items():\n",
    "    table.append({\n",
    "        \"modèle\": model_id,\n",
    "        \"F1 en domaine\": f\"{result['in_domain_f1_mean']:.3f}\",\n",
    "        \"F1 CaFE [IC 95 % acteurs]\": f\"{result['macro_f1']:.3f} [{result['macro_f1_ci'][0]:.3f} ; {result['macro_f1_ci'][1]:.3f}]\",\n",
    "        \"chute\": f\"{result['f1_drop']:+.3f}\",\n",
    "        \"accuracy\": f\"{result['accuracy']:.3f}\",\n",
    "        \"ECE\": f\"{result['ece_calibrated']:.3f}\",\n",
    "    })\n",
    "display(pd.DataFrame(table))\n",
    "display(pd.DataFrame({m: r[\"f1_by_intensity\"] for m, r in results.items()}).round(3))\n",
    "display(pd.DataFrame({m: r[\"f1_by_gender\"] for m, r in results.items()}).round(3))\n",
    "display(pd.DataFrame({m: r[\"f1_per_class\"] for m, r in results.items()}, index=class_names).round(3))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## j. Test de H2\n",
    "\n",
    "Principal : écart de F1 macro **mHuBERT − wav2vec 2.0**, bootstrap apparié sur les acteurs (mêmes\n",
    "acteurs, mêmes extraits à chaque tirage). Complément : McNemar **exact** sur les extraits ; il suppose\n",
    "des extraits indépendants, ce qui n'est pas vrai ici (78 extraits par acteur, 12 acteurs) : sa\n",
    "p-valeur est trop optimiste, elle ne sert que de repère."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "from echolens.training.evaluation import mcnemar_test\n",
    "\n",
    "first, second = ood.h2.first, ood.h2.second\n",
    "difference, low, high = paired_difference(\n",
    "    cafe_targets, predictions[first], predictions[second], samples, evaluation_config\n",
    ")\n",
    "verdict = h2_verdict(low, high)\n",
    "always_exact = evaluation_config.model_copy(update={\"mcnemar_exact_below\": 10**9})\n",
    "mcnemar = mcnemar_test(cafe_targets, predictions[first], predictions[second], always_exact)\n",
    "h2 = {\n",
    "    \"hypothesis\": \"mHuBERT dépasse wav2vec 2.0 anglais sur le français\",\n",
    "    \"difference_f1\": difference,\n",
    "    \"difference_ci\": [low, high],\n",
    "    \"verdict\": verdict,\n",
    "    \"mcnemar_exact\": {\n",
    "        \"both_right\": mcnemar.both_right,\n",
    "        f\"only_{first}_right\": mcnemar.only_a_right,\n",
    "        f\"only_{second}_right\": mcnemar.only_b_right,\n",
    "        \"both_wrong\": mcnemar.both_wrong,\n",
    "        \"p_value\": mcnemar.p_value,\n",
    "    },\n",
    "    \"mcnemar_note\": \"McNemar suppose des extraits indépendants : faux ici (12 acteurs, 78 extraits chacun). Indicatif seulement.\",\n",
    "}\n",
    "print(f\"{first} − {second} : écart de F1 {difference:+.3f}, IC 95 % [{low:+.3f} ; {high:+.3f}] → H2 {verdict}\")\n",
    "print(\"McNemar exact :\", h2[\"mcnemar_exact\"])"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## k. Résultats, figures et tableau pour la PR\n",
    "\n",
    "`c25_cafe_ood.json` (→ `training/results/`) ; les matrices de confusion du meilleur modèle de chaque\n",
    "famille sont dessinées par `training/scripts/c25_figures.py` (→ `report/figures/`), jamais à la main."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "import datetime\n",
    "import subprocess\n",
    "\n",
    "commit = subprocess.run(\n",
    "    [\"git\", \"-C\", CLONE_DIR, \"rev-parse\", \"HEAD\"], capture_output=True, text=True, check=True\n",
    ").stdout.strip()\n",
    "content = {\n",
    "    \"experiment\": \"c25_cafe_ood\",\n",
    "    \"date\": datetime.datetime.now(datetime.UTC).isoformat(),\n",
    "    \"branch\": BRANCH,\n",
    "    \"commit\": commit,\n",
    "    \"gpu\": torch.cuda.get_device_name(0),\n",
    "    \"torch_version\": torch.__version__,\n",
    "    \"classes\": class_names,\n",
    "    \"cafe\": {\n",
    "        \"clips\": len(cafe_rows),\n",
    "        \"actors\": len(set(cafe_actors)),\n",
    "        \"clips_per_class\": np.bincount(cafe_targets, minlength=class_count).tolist(),\n",
    "    },\n",
    "    \"bootstrap\": {\n",
    "        \"unit\": \"actor\",\n",
    "        \"draws\": evaluation_config.n_bootstrap,\n",
    "        \"seed\": evaluation_config.seed,\n",
    "        \"confidence\": evaluation_config.confidence,\n",
    "    },\n",
    "    \"in_domain_check\": check_rows,\n",
    "    \"models\": results,\n",
    "    \"h2\": h2,\n",
    "    \"runtime_s\": round(time.time() - NOTEBOOK_START),\n",
    "}\n",
    "OUTPUT_DIR = Path(\"/kaggle/working\")\n",
    "json_path = OUTPUT_DIR / \"c25_cafe_ood.json\"\n",
    "text = json.dumps(content, indent=2, ensure_ascii=False, default=float) + \"\\n\"\n",
    "json_path.write_text(text, encoding=\"utf-8\")\n",
    "# Le script de figures lit le JSON dans le dépôt cloné : on l'y copie, puis on récupère les PNG.\n",
    "(Path(CLONE_DIR) / \"training\" / \"results\").mkdir(parents=True, exist_ok=True)\n",
    "(Path(CLONE_DIR) / \"training\" / \"results\" / \"c25_cafe_ood.json\").write_text(text, encoding=\"utf-8\")\n",
    "!cd {CLONE_DIR} && python training/scripts/c25_figures.py\n",
    "for png in sorted((Path(CLONE_DIR) / \"report\" / \"figures\").glob(\"c25_cafe_confusion_*.png\")):\n",
    "    shutil.copy(png, OUTPUT_DIR / png.name)\n",
    "print(sorted(path.name for path in OUTPUT_DIR.iterdir()))"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "# Tableau et verdict au format de la description de PR (à copier dans la PR).\n",
    "lines = [\"| Modèle | F1 en domaine | F1 CaFE [IC 95 %, acteurs] | Chute |\", \"| --- | --- | --- | --- |\"]\n",
    "for model_id, result in results.items():\n",
    "    low_f1, high_f1 = result[\"macro_f1_ci\"]\n",
    "    lines.append(\n",
    "        f\"| {model_id} | {result['in_domain_f1_mean']:.3f} | \"\n",
    "        f\"{result['macro_f1']:.3f} [{low_f1:.3f} ; {high_f1:.3f}] | {result['f1_drop']:+.3f} |\"\n",
    "    )\n",
    "lines.append(\"\")\n",
    "lines.append(f\"**H2 ({verdict})** : mHuBERT − wav2vec 2.0 = {difference:+.3f}, IC 95 % [{low:+.3f} ; {high:+.3f}] (bootstrap apparié sur les 12 acteurs).\")\n",
    "lines.append(\"\")\n",
    "lines.append(\"Classes qui s'effondrent (F1 CaFE < 0,25) :\")\n",
    "for model_id, result in results.items():\n",
    "    collapsed = []\n",
    "    for name, value in zip(class_names, result[\"f1_per_class\"], strict=True):\n",
    "        if value < 0.25:\n",
    "            collapsed.append(f\"{name} ({value:.2f})\")\n",
    "    lines.append(f\"- {model_id} : {', '.join(collapsed) if collapsed else 'aucune'}\")\n",
    "pr_table = \"\\n\".join(lines)\n",
    "(OUTPUT_DIR / \"c25_pr_table.md\").write_text(pr_table + \"\\n\", encoding=\"utf-8\")\n",
    "print(pr_table)"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "shutil.rmtree(DOWNLOAD_DIR, ignore_errors=True)\n",
    "shutil.rmtree(CLONE_DIR, ignore_errors=True)\n",
    "print(sorted(path.name for path in Path(\"/kaggle/working\").iterdir()))"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}